# Voces y documentos peruanos · Evaluación

**Fecha: 25 de octubre de 2026, Perú.** Fuentes: corpus real de la Parte 1, Wikimedia Commons/BNP y OpenSLR 73, con atribuciones en los manifiestos. Modelos: PaddleOCR 3.7.0 vs. EasyOCR 1.7.2 (CER), faster-whisper `small` CPU vs. Qwen3-ASR 0.6B GPU (WER y latencia). LLM: Ollama local `qwen2.5:3b`.

La muestra de 10 documentos se estratifica 3 impresos/3 fotos/4 manuscritos y maximiza fuentes distintas. La muestra de 10 audios incluye las dos entrevistas naturales y ocho clips OpenSLR. Semilla: 20261025. Se guardan hashes de los archivos para comprobar que todos los modelos usan los mismos casos. Los parámetros se fijaron antes de consultar métricas.

**No se consideran referencias manuales ni auditoría humana las respuestas generadas por IA. Cuando falten, este notebook lo indica y no produce métricas finales.**

In [ ]:
import json, pandas as pd
from pathlib import Path
from IPython.display import display
import plotly.express as px
from evaluacion import *
configuracion = json.loads((BASE/'modelos/configuracion.json').read_text(encoding='utf-8'))
display(configuracion)
muestra = preparar_muestra()
display(muestra[['id','modalidad','tipo','sha256_archivo']])
print('Referencias manuales completas:', int(muestra.texto_referencia.str.strip().ne('').sum()), '/20')


## Inferencia separada de las referencias

Las predicciones se generan sin mirar referencias manuales ni métricas y se conservan como borradores privados. Solo la posterior puntuación crea los resultados finales. La latencia mide el consumo completo del generador de Whisper, excluye la carga del modelo y registra CPU/GPU. La comparación no aísla el efecto del hardware.

In [ ]:
RECALCULAR_MODELOS = False
if RECALCULAR_MODELOS:
    generar_predicciones('audio')
    generar_predicciones('documento')
predicciones = leer_csv(PRED, PRED_COLS)
if not predicciones.empty:
    tiempos = predicciones.assign(latencia_seg=pd.to_numeric(predicciones.latencia_seg)).groupby(['modalidad','modelo','dispositivo']).agg(casos=('id','nunique'), latencia_total_seg=('latencia_seg','sum')).reset_index()
    display(tiempos)
else:
    print('Todavía no hay predicciones de comparación guardadas.')


## CER y WER sobre las mismas referencias humanas

Normalización idéntica: Unicode NFC, minúsculas, signos a espacios, espacios compactados, conservación de tildes y exclusión de etiquetas `[H1]`/`[HABLANTE_1]`. Se suman sustituciones, inserciones y omisiones sobre todos los casos y se dividen entre los caracteres/palabras totales de referencia. No se promedian tasas individuales. `RTF = latencia / duración`; se informa también el RTF global ponderado por duración.

Las distancias se calculan con los originales locales antes de seudonimizar. Después se protegen los textos de exportación. Los conteos `errores` y `unidades_referencia` permiten recomponer las tasas agregadas sin publicar nombres. El diccionario de privacidad no forma parte de la normalización de evaluación.

In [ ]:
resumen = pd.DataFrame()
try:
    resumen = puntuar_predicciones()
    display(resumen)
except ValueError as pendiente:
    print('EVALUACIÓN FINAL PENDIENTE:', pendiente)


In [ ]:
if not resumen.empty:
    for metrica in ['CER','WER']:
        z=resumen[resumen.metrica==metrica]
        display(px.bar(z,x='modelo',y='valor',text='valor',title=f'{metrica}: comparación sobre las mismas 10 referencias'))
else:
    print('No hay CER/WER finales hasta completar las 20 referencias manuales.')


## Extracción estructurada y auditoría de 20 casos

Se exige modalidad, tema y resumen. Emisor/fecha/monto y citas/número de hablantes solo pueden aparecer si están sustentados; los ausentes deben ser `null`. La inferencia local puede preceder a la revisión, pero sus borradores no se publican hasta la validación humana. La auditoría usa una muestra fija de 10 documentos y 10 audios y cuenta como correcto solo el caso cuyos campos sean todos correctos.

In [ ]:
RECALCULAR_LLM = False
if RECALCULAR_LLM: extraer_llm('ollama')
ext = leer_csv(EXTRACCIONES, ['id','modalidad','campos_json','modelo_llm','fecha'])
print('Extracciones LLM reales:', len(ext))
if not ext.empty:
    display(ext.groupby(['modalidad','modelo_llm']).size().rename('extracciones').reset_index())
    auditoria = preparar_auditoria()
    display(auditoria[['id','modalidad','completamente_correcto','observacion_auditor']])
try:
    n,total,acuerdo=acuerdo_llm()
    print(f'Acuerdo LLM-humano: {n}/{total} = {acuerdo:.1%}')
except (ValueError,FileNotFoundError) as pendiente:
    print('AUDITORÍA HUMANA PENDIENTE:', pendiente)


## Casos difíciles y decisión

Las fotos incorporan reflejos, sombras y perspectiva; los manuscritos incluyen caligrafía histórica y tinta del reverso. Los audios naturales contienen ruido callejero. El análisis de precisión por tipo solo se completa con referencias manuales, conservando los mismos IDs para todos los modelos. La decisión debe considerar CER/WER, latencia, memoria y el coste de mantenimiento; no se exige que el alternativo gane.

In [ ]:
r=leer_csv(MODELOS, PRED_COLS+['cer','wer'])
if not r.empty:
    d=leer_csv(DOC_CSV)
    por_tipo=r[r.modalidad=='documento'].merge(d[['id','tipo']],on='id',validate='many_to_one')
    display(por_tipo[['id','tipo','modelo','cer','latencia_seg']])
    grupos=por_tipo.groupby(['tipo','modelo'])[['errores','unidades_referencia']].agg(lambda x:pd.to_numeric(x).sum())
    grupos['CER']=grupos.errores/grupos.unidades_referencia
    display(grupos)
    display(r[r.modalidad=='audio'][['id','modelo','wer','latencia_seg','rtf','dispositivo']])
else:
    print('Reflexión de precisión y elección final pendientes de CER/WER reales; no se inventa una conclusión.')


### Decisión con los resultados actuales

Se mantiene PaddleOCR y Whisper small como motores principales. En los originales y referencias disponibles, PaddleOCR tuvo CER 21,18 % frente a 40,11 % de EasyOCR, con mayor latencia total (657 frente a 394 s). El coste de esa precisión es más tiempo de CPU. Whisper small tuvo WER 40,95 % frente a 44,30 % de Qwen3-ASR y RTF global 0,113 frente a 0,118; la comparación mezcla CPU y GPU y no acredita superioridad universal.

El CER más alto se concentró en manuscritos. En `clip_021`, las inserciones elevaron WER por encima del 100 %: no se recorta esta tasa. Se pidió al estudiante una comprobación final de ambas entrevistas para descartar referencias abreviadas. Los resultados se recalculan si la referencia humana cambia; no se ajustan parámetros después de observar el test. Memoria máxima por proceso no medida. La elección final debe revisarse si las correcciones cambian estos resultados.

In [ ]:
from validar_publicacion import verificar
try:
    verificar()
    print('Textos revisados y aptos para preparar publicación. Comprobar también ejemplos y evidencias.')
except ValueError as pendiente:
    print('PUBLICACIÓN PENDIENTE:', pendiente)
